**install libraries**

In [1]:
!pip install xgboost
!pip install lightgbm
!pip install mlflow
!pip install optuna
!pip install -q kaggle kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**Configure Kaggle credentials.**

In [2]:

import os
import json

# Your Kaggle username
KAGGLE_USERNAME = 'mariamkapanadze'
# Your Kaggle API key (provided in previous context)
KAGGLE_API_KEY = 'KGAT_cd47955638de00296ee00cc490a5d931'

# Create the .kaggle directory if it doesn't exist
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True);

# Create kaggle.json file
kaggle_json_path = os.path.join(kaggle_dir, 'kaggle.json')
kaggle_config = {"username": KAGGLE_USERNAME, "key": KAGGLE_API_KEY}

with open(kaggle_json_path, 'w') as f:
    json.dump(kaggle_config, f)

# Set permissions for the kaggle.json file
os.chmod(kaggle_json_path, 0o600)
!cat ~/.kaggle/kaggle.json

!pip install -q kagglehub
import kagglehub
kagglehub.login()



{"username": "mariamkapanadze", "key": "KGAT_cd47955638de00296ee00cc490a5d931"}

Kaggle credentials set.
Kaggle credentials successfully validated.


**Download the competition**

In [3]:
path = kagglehub.competition_download(
    "walmart-recruiting-store-sales-forecasting"
)

print(path)

100%|██████████| 2.70M/2.70M [00:01<00:00, 2.37MB/s]

Extracting files...
/root/.cache/kagglehub/competitions/walmart-recruiting-store-sales-forecasting


**Load the CSV files.**

In [4]:
import zipfile
import os

for file in os.listdir(path):
    if file.endswith(".zip"):
        file_path = os.path.join(path, file)

        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            zip_ref.extractall(path)

print(os.listdir(path))

['test.csv.zip', 'train.csv', 'train.csv.zip', 'sampleSubmission.csv', 'test.csv', 'stores.csv', 'sampleSubmission.csv.zip', 'features.csv.zip', 'features.csv']


In [5]:
import pandas as pd

train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))
features = pd.read_csv(os.path.join(path, "features.csv"))
stores = pd.read_csv(os.path.join(path, "stores.csv"))
submission = pd.read_csv(os.path.join(path, "sampleSubmission.csv"))

**Display their shapes and first few rows.**

In [6]:
print(train.shape)
print(test.shape)
print(stores.shape)
print(features.shape)

(421570, 5)
(115064, 4)
(45, 3)
(8190, 12)


convert date

In [7]:
train["Date"] = pd.to_datetime(train["Date"])
test["Date"] = pd.to_datetime(test["Date"])
features["Date"] = pd.to_datetime(features["Date"])

merge everything into one dataset

In [8]:
train_merged = train.merge(stores, on="Store", how="left")
train_merged = train_merged.merge(features, on=["Store", "Date"], how="left")

test_merged = test.merge(stores, on="Store", how="left")
test_merged = test_merged.merge(features, on=["Store", "Date"], how="left")

sort by time

In [9]:

train_merged = train_merged.sort_values(["Store", "Dept", "Date"])
test_merged = test_merged.sort_values(["Store", "Dept", "Date"])

In [10]:
df = train_merged

df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month
df["week"] = df["Date"].dt.isocalendar().week.astype(int)
df["dayofweek"] = df["Date"].dt.dayofweek
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

In [11]:
print(train_merged.columns)

Index(['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday_x', 'Type', 'Size',
       'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3',
       'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'IsHoliday_y', 'year',
       'month', 'week', 'dayofweek', 'is_weekend'],
      dtype='object')


In [12]:
train_merged["IsHoliday"] = train_merged["IsHoliday_y"]
train_merged = train_merged.drop(columns=["IsHoliday_x", "IsHoliday_y"])


In [13]:
train_merged.columns

Index(['Store', 'Dept', 'Date', 'Weekly_Sales', 'Type', 'Size', 'Temperature',
       'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4',
       'MarkDown5', 'CPI', 'Unemployment', 'year', 'month', 'week',
       'dayofweek', 'is_weekend', 'IsHoliday'],
      dtype='object')

In [14]:
train_merged = train_merged.sort_values(["Store", "Dept", "Date"])

define target + feature and prepare dataset

In [15]:
target = "Weekly_Sales"

features = [
    "Store",
    "Dept",
    "IsHoliday",
    "Size",
    "Type",
    "Temperature",
    "Fuel_Price",
    "CPI",
    "Unemployment",
    "year",
    "month",
    "week",
    "dayofweek",
    "is_weekend"
]



In [16]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
train_merged["Type"] = le.fit_transform(train_merged["Type"])

In [17]:
X = train_merged[features]
y = train_merged[target]

**Evaluation metrics: MAE, WMAE (competition metric), and Bias**

In [18]:
import numpy as np

def calculate_wmae(y_true, y_pred, is_holiday):
    """Weighted Mean Absolute Error, matching the Walmart Kaggle competition
    metric: holiday weeks get a weight of 5, all other weeks a weight of 1."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    weights = np.where(np.asarray(is_holiday).astype(bool), 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

def calculate_bias(y_true, y_pred):
    """Mean signed error. Positive = model over-predicts on average,
    negative = model under-predicts on average."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(y_pred - y_true)


time based split

In [19]:
split_date = train_merged["Date"].max() - pd.Timedelta(weeks=31)

train_df = train_merged[train_merged["Date"] < split_date]
val_df = train_merged[train_merged["Date"] >= split_date]

X_train = train_df[features]
y_train = train_df[target]

X_val = val_df[features]
y_val = val_df[target]

train xgboost baseline

In [20]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

evaluate

In [21]:
from sklearn.metrics import mean_absolute_error

preds = model.predict(X_val)

mae = mean_absolute_error(y_val, preds)
wmae = calculate_wmae(y_val, preds, val_df["IsHoliday"])
bias = calculate_bias(y_val, preds)

print("Baseline MAE:", mae)
print("Baseline WMAE:", wmae)
print("Baseline Bias:", bias)


Baseline MAE: 3248.304756923445
Baseline WMAE: 3287.301943200054
Baseline Bias: 178.0114814563423


**wandb setup**

In [22]:
import wandb

wandb.init(
    project="walmart-sales-forecasting",
    name="baseline_xgboost",
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mkapa22 (mkapa22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


logging baseline

In [23]:
wandb.log({
    "mae": mae,
    "wmae": wmae,
    "bias": bias,
    "model": "XGBoost",
    "features": "basic_tabular_features",
    "split": "time_based_last_31_weeks"
})


In [24]:
wandb.finish()

bias,▁
mae,▁
wmae,▁
bias,178.01148
features,basic_tabular_featur...
mae,3248.30476
model,XGBoost
split,time_based_last_31_w...
wmae,3287.30194


sort correctly

In [25]:
train_merged = train_merged.sort_values(["Store", "Dept", "Date"])

create lag features

In [26]:
for lag in [1, 2, 4, 8, 52]:
    train_merged[f"lag_{lag}"] = (
        train_merged.groupby(["Store", "Dept"])["Weekly_Sales"]
        .shift(lag)
    )

In [27]:
train_merged[["Store", "Dept", "Date", "Weekly_Sales", "lag_1"]].head(10)

,Store,Dept,Date,Weekly_Sales,lag_1
0,1,1,2010-02-05,24924.50,NaN
1,1,1,2010-02-12,46039.49,24924.50
2,1,1,2010-02-19,41595.55,46039.49
3,1,1,2010-02-26,19403.54,41595.55
4,1,1,2010-03-05,21827.90,19403.54
5,1,1,2010-03-12,21043.39,21827.90
6,1,1,2010-03-19,22136.64,21043.39
7,1,1,2010-03-26,26229.21,22136.64
8,1,1,2010-04-02,57258.43,26229.21
9,1,1,2010-04-09,42960.91,57258.43


rolling features

In [28]:
train_merged["rolling_mean_4"] = (
    train_merged.groupby(["Store", "Dept"])["Weekly_Sales"]
    .shift(1)
    .rolling(4)
    .mean()
)

train_merged["rolling_mean_8"] = (
    train_merged.groupby(["Store", "Dept"])["Weekly_Sales"]
    .shift(1)
    .rolling(8)
    .mean()
)

In [29]:
train_model = train_merged.dropna()

update features

In [30]:
features = [
    "Store",
    "Dept",
    "IsHoliday",
    "Size",
    "Type",
    "Temperature",
    "Fuel_Price",
    "CPI",
    "Unemployment",
    "year",
    "month",
    "week",
    "dayofweek",
    "is_weekend",
    "lag_1",
    "lag_2",
    "lag_4",
    "lag_8",
    "lag_52",
    "rolling_mean_4",
    "rolling_mean_8"
]

model retraining

In [31]:
X = train_model[features]
y = train_model["Weekly_Sales"]
split_date = train_model["Date"].max() - pd.Timedelta(weeks=31)

train_df = train_model[train_model["Date"] < split_date]
val_df = train_model[train_model["Date"] >= split_date]

X_train = train_df[features]
y_train = train_df["Weekly_Sales"]

X_val = val_df[features]
y_val = val_df["Weekly_Sales"]

In [32]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=400,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=400,
             n_jobs=None, num_parallel_tree=None, ...)

evaluation

In [33]:
from sklearn.metrics import mean_absolute_error

preds = model.predict(X_val)
mae = mean_absolute_error(y_val, preds)
wmae = calculate_wmae(y_val, preds, val_df["IsHoliday"])
bias = calculate_bias(y_val, preds)

print("Lag features MAE:", mae)
print("Lag features WMAE:", wmae)
print("Lag features Bias:", bias)


Lag features MAE: 1708.292665540107
Lag features WMAE: 1793.4775843983327
Lag features Bias: 662.1076490449316


log new run

In [34]:
wandb.init(
    project="walmart-sales-forecasting",
    name="xgboost_with_lags"
)

wandb.log({
    "mae": mae,
    "wmae": wmae,
    "bias": bias,
    "features": "lags + rolling",
    "model": "XGBoost"
})

wandb.finish()


bias,▁
mae,▁
wmae,▁
bias,662.10765
features,lags + rolling
mae,1708.29267
model,XGBoost
wmae,1793.47758


run 3 / add rolling standart deviation

In [35]:
train_merged["rolling_std_4"] = (
    train_merged.groupby(["Store", "Dept"])["Weekly_Sales"]
    .shift(1)
    .rolling(4)
    .std()
)

train_merged["rolling_std_8"] = (
    train_merged.groupby(["Store", "Dept"])["Weekly_Sales"]
    .shift(1)
    .rolling(8)
    .std()
)

In [36]:
train_model = train_merged.dropna()

In [37]:
features = [
    "Store","Dept","IsHoliday","Size","Type",
    "Temperature","Fuel_Price","CPI","Unemployment",
    "year","month","week","dayofweek","is_weekend",
    "lag_1","lag_2","lag_4","lag_8","lag_52",
    "rolling_mean_4","rolling_mean_8",
    "rolling_std_4","rolling_std_8"
]


In [38]:
X = train_model[features]
y = train_model["Weekly_Sales"]

split_date = train_model["Date"].max() - pd.Timedelta(weeks=31)

train_df = train_model[train_model["Date"] < split_date]
val_df = train_model[train_model["Date"] >= split_date]

X_train = train_df[features]
y_train = train_df["Weekly_Sales"]

X_val = val_df[features]
y_val = val_df["Weekly_Sales"]

train

In [39]:
model = XGBRegressor(
    n_estimators=400,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=400,
             n_jobs=None, num_parallel_tree=None, ...)

evaluate

In [40]:
from sklearn.metrics import mean_absolute_error

preds = model.predict(X_val)
mae = mean_absolute_error(y_val, preds)
wmae = calculate_wmae(y_val, preds, val_df["IsHoliday"])
bias = calculate_bias(y_val, preds)

print(f"Run 3 MAE: {mae:.4f}")
print(f"Run 3 WMAE: {wmae:.4f}")
print(f"Run 3 Bias: {bias:.4f}")


Run 3 MAE: 1686.6303
Run 3 WMAE: 1749.5478
Run 3 Bias: 611.1443


In [41]:
import wandb

wandb.init(
    project="walmart-sales-forecasting",
    name="run3_rolling_std"
)

wandb.log({
    "mae": mae,
    "wmae": wmae,
    "bias": bias,
    "model": "XGBoost",
    "feature_group": "lags + rolling_mean + rolling_std"
})

wandb.finish()


bias,▁
mae,▁
wmae,▁
bias,611.14433
feature_group,lags + rolling_mean ...
mae,1686.63026
model,XGBoost
wmae,1749.54784


Add markdown feature




In [42]:
markdown_cols = [
    "MarkDown1","MarkDown2","MarkDown3",
    "MarkDown4","MarkDown5"
]

train_model = train_merged.dropna().copy()
features += markdown_cols

In [43]:
X = train_model[features]
y = train_model["Weekly_Sales"]

split_date = train_model["Date"].max() - pd.Timedelta(weeks=31)

train_df = train_model[train_model["Date"] < split_date]
val_df = train_model[train_model["Date"] >= split_date]

X_train = train_df[features]
y_train = train_df["Weekly_Sales"]
X_val = val_df[features]
y_val = val_df["Weekly_Sales"]

model = XGBRegressor(
    n_estimators=400,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

from sklearn.metrics import mean_absolute_error

preds = model.predict(X_val)
mae = mean_absolute_error(y_val, preds)

wmae = calculate_wmae(y_val, preds, val_df["IsHoliday"])
bias = calculate_bias(y_val, preds)

print(f"Run 4 MAE: {mae:.4f}")
print(f"Run 4 WMAE: {wmae:.4f}")
print(f"Run 4 Bias: {bias:.4f}")

Run 4 MAE: 1608.3042
Run 4 WMAE: 1609.7618
Run 4 Bias: 476.5750


In [44]:
wandb.init(
    project="walmart-sales-forecasting",
    name="run4_markdown features"
)

wandb.log({
    "mae": mae,
    "wmae": wmae,
    "bias": bias,
    "model": "XGBoost",
    "run": "run4_markdown features"
})

wandb.finish()


bias,▁
mae,▁
wmae,▁
bias,476.575
mae,1608.30415
model,XGBoost
run,run4_markdown featur...
wmae,1609.7618


run 5 add interaction feature

In [45]:
train_model["Store_Dept"] = (
    train_model["Store"].astype(str)
    + "_"
    + train_model["Dept"].astype(str)
)

In [46]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
train_model["Store_Dept"] = le.fit_transform(train_model["Store_Dept"])

In [47]:
features.append("Store_Dept")

In [48]:
X = train_model[features]
y = train_model["Weekly_Sales"]

In [49]:
split_date = train_model["Date"].max() - pd.Timedelta(weeks=31)

train_df = train_model[train_model["Date"] < split_date]
val_df = train_model[train_model["Date"] >= split_date]

X_train = train_df[features]
y_train = train_df["Weekly_Sales"]

X_val = val_df[features]
y_val = val_df["Weekly_Sales"]

In [50]:
from xgboost import XGBRegressor

model_run5 = XGBRegressor(
    n_estimators=400,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model_run5.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=400,
             n_jobs=None, num_parallel_tree=None, ...)

In [51]:
from sklearn.metrics import mean_absolute_error

preds = model_run5.predict(X_val)

mae_run5 = mean_absolute_error(
    y_val,
    preds
)
wmae_run5 = calculate_wmae(y_val, preds, val_df["IsHoliday"])
bias_run5 = calculate_bias(y_val, preds)

print("Run 5 MAE:", mae_run5)
print("Run 5 WMAE:", wmae_run5)
print("Run 5 Bias:", bias_run5)

import wandb

wandb.init(
    project="walmart-sales-forecasting",
    name="run5_store_department_feature"
)

wandb.log({
    "run": "Store_Department_interaction",
    "model": "XGBoost",
    "features": "lags + rolling + markdown + store_dept",
    "mae": mae_run5,
    "wmae": wmae_run5,
    "bias": bias_run5,
    "n_estimators": 400,
    "max_depth": 7,
    "learning_rate": 0.05
})

wandb.finish()


Run 5 MAE: 1605.7937768164336
Run 5 WMAE: 1614.4351098752106
Run 5 Bias: 446.15658926678225


bias,▁
learning_rate,▁
mae,▁
max_depth,▁
n_estimators,▁
wmae,▁
bias,446.15659
features,lags + rolling + mar...
learning_rate,0.05
mae,1605.79378
max_depth,7


In [52]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import wandb


model_run6 = XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    random_state=42
)

model_run6.fit(X_train, y_train)


preds = model_run6.predict(X_val)

mae_run6 = mean_absolute_error(
    y_val,
    preds
)
wmae_run6 = calculate_wmae(y_val, preds, val_df["IsHoliday"])
bias_run6 = calculate_bias(y_val, preds)

print("Run 6 MAE:", mae_run6)
print("Run 6 WMAE:", wmae_run6)
print("Run 6 Bias:", bias_run6)


wandb.init(
    project="walmart-sales-forecasting",
    name="run6_tuning_v1"
)

wandb.log({
    "run": "hyperparameter_tuning_v1",
    "model": "XGBoost",
    "mae": mae_run6,
    "wmae": wmae_run6,
    "bias": bias_run6,
    "n_estimators": 600,
    "max_depth": 8,
    "learning_rate": 0.05
})

wandb.finish()


Run 6 MAE: 1613.6121030085758
Run 6 WMAE: 1622.1151444504726
Run 6 Bias: 479.2599066942253


bias,▁
learning_rate,▁
mae,▁
max_depth,▁
n_estimators,▁
wmae,▁
bias,479.25991
learning_rate,0.05
mae,1613.6121
max_depth,8
model,XGBoost


In [53]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import wandb


model_run7 = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=10,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=3,
    gamma=0.1,
    random_state=42
)

model_run7.fit(X_train, y_train)


preds = model_run7.predict(X_val)

mae_run7 = mean_absolute_error(
    y_val,
    preds
)
wmae_run7 = calculate_wmae(y_val, preds, val_df["IsHoliday"])
bias_run7 = calculate_bias(y_val, preds)

print("Run 7 MAE:", mae_run7)
print("Run 7 WMAE:", wmae_run7)
print("Run 7 Bias:", bias_run7)


wandb.init(
    project="walmart-sales-forecasting",
    name="run7_tuning_v2"
)

wandb.log({
    "run": "hyperparameter_tuning_v2",
    "model": "XGBoost",
    "mae": mae_run7,
    "wmae": wmae_run7,
    "bias": bias_run7,
    "n_estimators": 800,
    "max_depth": 10,
    "learning_rate": 0.03
})

wandb.finish()


Run 7 MAE: 1595.2253320096995
Run 7 WMAE: 1606.1367655344634
Run 7 Bias: 454.51472678707324


bias,▁
learning_rate,▁
mae,▁
max_depth,▁
n_estimators,▁
wmae,▁
bias,454.51473
learning_rate,0.03
mae,1595.22533
max_depth,10
model,XGBoost


run 8 regularization

In [54]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import wandb


model_run8 = XGBRegressor(
    n_estimators=700,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=1,       # L1 regularization
    reg_lambda=5,      # L2 regularization
    random_state=42
)


model_run8.fit(
    X_train,
    y_train
)


preds = model_run8.predict(X_val)


mae_run8 = mean_absolute_error(
    y_val,
    preds
)
wmae_run8 = calculate_wmae(y_val, preds, val_df["IsHoliday"])
bias_run8 = calculate_bias(y_val, preds)

print("Run 8 MAE:", mae_run8)
print("Run 8 WMAE:", wmae_run8)
print("Run 8 Bias:", bias_run8)


wandb.init(
    project="walmart-sales-forecasting",
    name="run8_regularized_xgboost"
)


wandb.log({
    "run": "regularized_model",
    "model": "XGBoost",
    "mae": mae_run8,
    "wmae": wmae_run8,
    "bias": bias_run8,
    "n_estimators": 700,
    "learning_rate": 0.05,
    "max_depth": 7,
    "reg_alpha": 1,
    "reg_lambda": 5
})


wandb.finish()


Run 8 MAE: 1600.363923058747
Run 8 WMAE: 1602.7977876053205
Run 8 Bias: 433.68509834417165


bias,▁
learning_rate,▁
mae,▁
max_depth,▁
n_estimators,▁
reg_alpha,▁
reg_lambda,▁
wmae,▁
bias,433.6851
learning_rate,0.05
mae,1600.36392


run 9 lower learning rate

In [55]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import wandb


model_run9 = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.01,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    random_state=42
)


model_run9.fit(
    X_train,
    y_train
)


preds = model_run9.predict(X_val)


mae_run9 = mean_absolute_error(
    y_val,
    preds
)
wmae_run9 = calculate_wmae(y_val, preds, val_df["IsHoliday"])
bias_run9 = calculate_bias(y_val, preds)

print("Run 9 MAE:", mae_run9)
print("Run 9 WMAE:", wmae_run9)
print("Run 9 Bias:", bias_run9)


wandb.init(
    project="walmart-sales-forecasting",
    name="run9_low_learning_rate"
)


wandb.log({
    "run": "low_learning_rate_model",
    "model": "XGBoost",
    "mae": mae_run9,
    "wmae": wmae_run9,
    "bias": bias_run9,
    "n_estimators": 1200,
    "learning_rate": 0.01,
    "max_depth": 8
})


wandb.finish()


Run 9 MAE: 1583.9048247084975
Run 9 WMAE: 1590.3480697528914
Run 9 Bias: 432.24945148093997


bias,▁
learning_rate,▁
mae,▁
max_depth,▁
n_estimators,▁
wmae,▁
bias,432.24945
learning_rate,0.01
mae,1583.90482
max_depth,8
model,XGBoost


In [56]:
model_run10 = XGBRegressor(
    n_estimators=1500,
    learning_rate=0.01,
    max_depth=8,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=3,
    reg_alpha=0.5,
    reg_lambda=3,
    random_state=42
)

model_run10.fit(
    X_train,
    y_train
)


preds = model_run10.predict(X_val)


mae_run10 = mean_absolute_error(
    y_val,
    preds
)
wmae_run10 = calculate_wmae(y_val, preds, val_df["IsHoliday"])
bias_run10 = calculate_bias(y_val, preds)

print("Run 10 MAE:", mae_run10)
print("Run 10 WMAE:", wmae_run10)
print("Run 10 Bias:", bias_run10)


wandb.init(
    project="walmart-sales-forecasting",
    name="run10"
)


wandb.log({
    "run": "run10",
    "model": "XGBoost",
    "mae": mae_run10,
    "wmae": wmae_run10,
    "bias": bias_run10,
    "n_estimators": 1500,
    "learning_rate": 0.01,
    "max_depth": 8
})


wandb.finish()


Run 10 MAE: 1582.5702328955829
Run 10 WMAE: 1586.249640423206
Run 10 Bias: 417.71608401760284


bias,▁
learning_rate,▁
mae,▁
max_depth,▁
n_estimators,▁
wmae,▁
bias,417.71608
learning_rate,0.01
mae,1582.57023
max_depth,8
model,XGBoost


In [57]:
import joblib

joblib.dump(
    model_run10,
    "xgboost_best_model.pkl"
)

['xgboost_best_model.pkl']

In [58]:
import wandb

run = wandb.init(
    project="walmart-sales-forecasting",
    name="xgboost_final_model"
)

wandb.log({
    "final_mae": mae_run10,
    "final_wmae": wmae_run10,
    "final_bias": bias_run10,
    "model": "XGBoost",
    "features": "lags + rolling + markdown + tuned parameters"
})

artifact = wandb.Artifact(
    "xgboost-best-model",
    type="model"
)

artifact.add_file(
    "xgboost_best_model.pkl"
)

run.log_artifact(artifact)

wandb.finish()


wandb: WARNING Artifact "xgboost-best-model" already exists with the same content. No new version will be created.


final_bias,▁
final_mae,▁
final_wmae,▁
features,lags + rolling + mar...
final_bias,417.71608
final_mae,1582.57023
final_wmae,1586.24964
model,XGBoost


## Run 11: Train + Stores only (no `features.csv`)

This run drops every column that came from `features.csv` (Temperature, Fuel_Price, CPI, Unemployment, MarkDown1-5) and only uses `train.csv` merged with `stores.csv`, plus date-derived and lag/rolling features built from `Weekly_Sales` itself.

In [59]:
train_ts = train.merge(stores, on="Store", how="left")

train_ts["Date"] = pd.to_datetime(train_ts["Date"])
train_ts = train_ts.sort_values(["Store", "Dept", "Date"])

print(train_ts.shape)
train_ts.head()


(421570, 7)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size
0,1,1,2010-02-05,24924.50,False,A,151315
1,1,1,2010-02-12,46039.49,True,A,151315
2,1,1,2010-02-19,41595.55,False,A,151315
3,1,1,2010-02-26,19403.54,False,A,151315
4,1,1,2010-03-05,21827.90,False,A,151315


Create date features

In [60]:
train_ts["year"] = train_ts["Date"].dt.year
train_ts["month"] = train_ts["Date"].dt.month
train_ts["week"] = train_ts["Date"].dt.isocalendar().week.astype(int)
train_ts["dayofweek"] = train_ts["Date"].dt.dayofweek
train_ts["is_weekend"] = (train_ts["dayofweek"] >= 5).astype(int)


Encode `Type`

In [61]:
from sklearn.preprocessing import LabelEncoder

le_ts = LabelEncoder()
train_ts["Type"] = le_ts.fit_transform(train_ts["Type"])


Lag + rolling features (built only from `Weekly_Sales`, no features.csv columns)

In [62]:
for lag in [1, 2, 4, 8, 52]:
    train_ts[f"lag_{lag}"] = (
        train_ts.groupby(["Store", "Dept"])["Weekly_Sales"]
        .shift(lag)
    )

train_ts["rolling_mean_4"] = (
    train_ts.groupby(["Store", "Dept"])["Weekly_Sales"]
    .shift(1)
    .rolling(4)
    .mean()
)

train_ts["rolling_mean_8"] = (
    train_ts.groupby(["Store", "Dept"])["Weekly_Sales"]
    .shift(1)
    .rolling(8)
    .mean()
)

train_ts_model = train_ts.dropna()


Define features (train + stores only) and prepare dataset

In [63]:
features_ts = [
    "Store",
    "Dept",
    "IsHoliday",
    "Size",
    "Type",
    "year",
    "month",
    "week",
    "dayofweek",
    "is_weekend",
    "lag_1",
    "lag_2",
    "lag_4",
    "lag_8",
    "lag_52",
    "rolling_mean_4",
    "rolling_mean_8"
]

target = "Weekly_Sales"

X = train_ts_model[features_ts]
y = train_ts_model[target]


Time based split

In [64]:
split_date_ts = train_ts_model["Date"].max() - pd.Timedelta(weeks=31)

train_df_ts = train_ts_model[train_ts_model["Date"] < split_date_ts]
val_df_ts = train_ts_model[train_ts_model["Date"] >= split_date_ts]

X_train_ts = train_df_ts[features_ts]
y_train_ts = train_df_ts[target]

X_val_ts = val_df_ts[features_ts]
y_val_ts = val_df_ts[target]


Train XGBoost (train + stores only)

In [65]:
from xgboost import XGBRegressor

model_ts = XGBRegressor(
    n_estimators=400,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model_ts.fit(X_train_ts, y_train_ts)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=400,
             n_jobs=None, num_parallel_tree=None, ...)

Evaluate

In [66]:
from sklearn.metrics import mean_absolute_error

preds_ts = model_ts.predict(X_val_ts)

mae_ts = mean_absolute_error(y_val_ts, preds_ts)
wmae_ts = calculate_wmae(y_val_ts, preds_ts, val_df_ts["IsHoliday"])
bias_ts = calculate_bias(y_val_ts, preds_ts)

print("Train+Stores-only MAE:", mae_ts)
print("Train+Stores-only WMAE:", wmae_ts)
print("Train+Stores-only Bias:", bias_ts)


Train+Stores-only MAE: 1359.011510156509
Train+Stores-only WMAE: 1367.3811840155774
Train+Stores-only Bias: 128.3691891303493


Log run to wandb

In [67]:
import wandb

wandb.init(
    project="walmart-sales-forecasting",
    name="run11_train_stores_only"
)

wandb.log({
    "run": "train_stores_only_no_features_csv",
    "model": "XGBoost",
    "features": "store/dept/holiday/size/type + date + lags/rolling (no features.csv)",
    "mae": mae_ts,
    "wmae": wmae_ts,
    "bias": bias_ts,
    "n_estimators": 400,
    "max_depth": 7,
    "learning_rate": 0.05
})

wandb.finish()


bias,▁
learning_rate,▁
mae,▁
max_depth,▁
n_estimators,▁
wmae,▁
bias,128.36919
features,store/dept/holiday/s...
learning_rate,0.05
mae,1359.01151
max_depth,7


Save best model as a Pipeline + register in MLflow Model Registry

In [68]:
features_df_raw = pd.read_csv(os.path.join(path, "features.csv"))
features_df_raw["Date"] = pd.to_datetime(features_df_raw["Date"])

In [69]:
CANDIDATES = {
    "run5":  dict(model=model_run5,  wmae=wmae_run5,  mae=mae_run5,  bias=bias_run5,
                  use_features_csv=True,  markdown=True,  rolling_std=True,  store_dept=True,  columns=list(features)),
    "run6":  dict(model=model_run6,  wmae=wmae_run6,  mae=mae_run6,  bias=bias_run6,
                  use_features_csv=True,  markdown=True,  rolling_std=True,  store_dept=True,  columns=list(features)),
    "run7":  dict(model=model_run7,  wmae=wmae_run7,  mae=mae_run7,  bias=bias_run7,
                  use_features_csv=True,  markdown=True,  rolling_std=True,  store_dept=True,  columns=list(features)),
    "run8":  dict(model=model_run8,  wmae=wmae_run8,  mae=mae_run8,  bias=bias_run8,
                  use_features_csv=True,  markdown=True,  rolling_std=True,  store_dept=True,  columns=list(features)),
    "run9":  dict(model=model_run9,  wmae=wmae_run9,  mae=mae_run9,  bias=bias_run9,
                  use_features_csv=True,  markdown=True,  rolling_std=True,  store_dept=True,  columns=list(features)),
    "run10": dict(model=model_run10, wmae=wmae_run10, mae=mae_run10, bias=bias_run10,
                  use_features_csv=True,  markdown=True,  rolling_std=True,  store_dept=True,  columns=list(features)),
    "run11_train_stores_only": dict(model=model_ts, wmae=wmae_ts, mae=mae_ts, bias=bias_ts,
                  use_features_csv=False, markdown=False, rolling_std=False, store_dept=False, columns=list(features_ts)),
}

comparison = pd.DataFrame({
    name: {"MAE": cfg["mae"], "WMAE": cfg["wmae"], "Bias": cfg["bias"]}
    for name, cfg in CANDIDATES.items()
}).T.sort_values("WMAE")

print(comparison)

best_name = comparison.index[0]
best_cfg = CANDIDATES[best_name]
print(f"\nBest model: {best_name}  (WMAE={best_cfg['wmae']:.4f})")


                                 MAE         WMAE        Bias
run11_train_stores_only  1359.011510  1367.381184  128.369189
run10                    1582.570233  1586.249640  417.716084
run9                     1583.904825  1590.348070  432.249451
run8                     1600.363923  1602.797788  433.685098
run7                     1595.225332  1606.136766  454.514727
run5                     1605.793777  1614.435110  446.156589
run6                     1613.612103  1622.115144  479.259907

Best model: run11_train_stores_only  (WMAE=1367.3812)


In [76]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline


class MergeStaticData(BaseEstimator, TransformerMixin):
    """Merges raw (Store, Dept, Date, IsHoliday[, Weekly_Sales]) rows with
    stores.csv and, optionally, features.csv -- exactly like the earlier
    merge cells in this notebook, but wrapped as a transformer."""

    def __init__(self, stores_df, features_df=None, use_features=False):
        self.stores_df = stores_df
        self.features_df = features_df
        self.use_features = use_features

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df["Date"] = pd.to_datetime(df["Date"])
        df = df.merge(self.stores_df, on="Store", how="left")

        if self.use_features and self.features_df is not None:
            df = df.merge(self.features_df, on=["Store", "Date"], how="left")
            if "IsHoliday_y" in df.columns:
                df["IsHoliday"] = df["IsHoliday_y"]
                df = df.drop(columns=["IsHoliday_x", "IsHoliday_y"])
        return df


class DateFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df["year"] = df["Date"].dt.year
        df["month"] = df["Date"].dt.month
        df["week"] = df["Date"].dt.isocalendar().week.astype(int)
        df["dayofweek"] = df["Date"].dt.dayofweek
        df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
        return df


class TypeEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.encoder_ = LabelEncoder().fit(X["Type"])
        return self

    def transform(self, X):
        df = X.copy()
        df["Type"] = self.encoder_.transform(df["Type"])
        return df


class LagRollingFeatures(BaseEstimator, TransformerMixin):
    """Builds lag / rolling-mean / rolling-std features for Weekly_Sales
    using a Store/Dept/Date history captured from the TRAINING data at fit
    time -- so it can be applied to a test set that has no Weekly_Sales
    column at all, exactly like a real deployment pipeline would need to."""

    def __init__(self, lags=(1, 2, 4, 8, 52), roll_windows=(4, 8),
                 add_std=False, add_store_dept=False):
        self.lags = list(lags)
        self.roll_windows = list(roll_windows)
        self.add_std = add_std
        self.add_store_dept = add_store_dept

    def fit(self, X, y=None):
        hist = X[["Store", "Dept", "Date", "Weekly_Sales"]].copy()
        hist["Date"] = pd.to_datetime(hist["Date"])
        self.history_df_ = hist.drop_duplicates(subset=["Store", "Dept", "Date"])

        if self.add_store_dept:
            key = X["Store"].astype(str) + "_" + X["Dept"].astype(str)
            self.sd_classes_ = sorted(key.unique())
            self.sd_map_ = {c: i for i, c in enumerate(self.sd_classes_)}
        return self

    def _lag_lookup(self, df, weeks_back):
        key = df[["Store", "Dept"]].copy()
        key["Date"] = df["Date"] - pd.Timedelta(weeks=weeks_back)
        merged = key.merge(self.history_df_, on=["Store", "Dept", "Date"], how="left")
        return merged["Weekly_Sales"].values

    def transform(self, X):
        df = X.copy()
        df["Date"] = pd.to_datetime(df["Date"])

        lag_values = {}
        for lag in self.lags:
            vals = self._lag_lookup(df, lag)
            df[f"lag_{lag}"] = vals
            lag_values[lag] = vals

        max_window = max(self.roll_windows) if self.roll_windows else 0
        for w in range(1, max_window + 1):
            if w not in lag_values:
                lag_values[w] = self._lag_lookup(df, w)

        import warnings

        for w in self.roll_windows:
            window_vals = np.column_stack([lag_values[k] for k in range(1, w + 1)])
            with warnings.catch_warnings():
                # Rows far beyond the training history have no lag values at
                # all (all-NaN window) -- nanmean/nanstd correctly return NaN
                # there, this just silences the expected "empty slice" notice.
                warnings.simplefilter("ignore", category=RuntimeWarning)
                df[f"rolling_mean_{w}"] = np.nanmean(window_vals, axis=1)
                if self.add_std:
                    df[f"rolling_std_{w}"] = np.nanstd(window_vals, axis=1, ddof=1)

        if self.add_store_dept:
            key = df["Store"].astype(str) + "_" + df["Dept"].astype(str)
            df["Store_Dept"] = key.map(self.sd_map_).fillna(-1).astype(int)

        return df


class ColumnSelector(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X[self.columns]


In [77]:
merge_step = MergeStaticData(
    stores_df=stores,
    features_df=features_df_raw if best_cfg["use_features_csv"] else None,
    use_features=best_cfg["use_features_csv"]
)

date_step = DateFeatures()
type_step = TypeEncoder()
lag_step = LagRollingFeatures(
    lags=(1, 2, 4, 8, 52),
    roll_windows=(4, 8),
    add_std=best_cfg["rolling_std"],
    add_store_dept=best_cfg["store_dept"]
)
select_step = ColumnSelector(best_cfg["columns"])

# Raw train + stores [+ features] -> used only to FIT the stateful steps
# (TypeEncoder's LabelEncoder, LagRollingFeatures' history table)
fit_source = merge_step.transform(train)
fit_source = date_step.transform(fit_source)

type_step.fit(fit_source)
lag_step.fit(fit_source)

best_pipeline = Pipeline([
    ("merge_static", merge_step),
    ("date_features", date_step),
    ("encode_type", type_step),
    ("lag_rolling", lag_step),
    ("select_features", select_step),
    ("model", best_cfg["model"]),
])

best_pipeline


Pipeline(steps=[('merge_static',
                 MergeStaticData(stores_df=    Store Type    Size
0       1    A  151315
1       2    A  202307
2       3    B   37392
3       4    A  205863
4       5    B   34875
5       6    A  202505
6       7    B   70713
7       8    A  155078
8       9    B  125833
9      10    B  126512
10     11    A  207499
11     12    B  112238
12     13    A  219622
13     14    A  200898
14     15    B  123737
15     16    B   57197
16     17    B   93188
17     18    B  120653
18     19    A  203819
19     20    A  203742
20     21    B  140167
21     22    B  119557
22     23    B  114533
23     24    A  203819
24     25    B  128107
25     26    A  152513
26     27    A  204184
27     28    A  2...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=7, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=400, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [78]:
raw_test = pd.read_csv(os.path.join(path, "test.csv"))  # untouched, as downloaded

test_predictions = best_pipeline.predict(raw_test)

print("raw_test shape:", raw_test.shape)
print("predictions shape:", test_predictions.shape)
print(test_predictions[:10])


raw_test shape: (115064, 4)
predictions shape: (115064,)
[ 40038.836  35599.258  54441.062  55249.082  98681.836  99077.81
 102292.41  113793.125  97544.266  74885.67 ]


save the pipeline

In [79]:
import joblib

joblib.dump(best_pipeline, "best_model_pipeline.pkl")


['best_model_pipeline.pkl']

Register the winning Pipeline in the MLflow Model Registry

In [80]:
import mlflow
import mlflow.sklearn


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("walmart-sales-forecasting")

with mlflow.start_run(run_name=f"best_pipeline_{best_name}"):
    mlflow.log_param("source_run", best_name)
    mlflow.log_param("use_features_csv", best_cfg["use_features_csv"])
    mlflow.log_param("markdown_features", best_cfg["markdown"])
    mlflow.log_param("store_dept_feature", best_cfg["store_dept"])

    mlflow.log_metric("mae", best_cfg["mae"])
    mlflow.log_metric("wmae", best_cfg["wmae"])
    mlflow.log_metric("bias", best_cfg["bias"])

    mlflow.sklearn.log_model(
        sk_model=best_pipeline,
        name="model",
        registered_model_name="walmart-sales-xgboost-pipeline",
        serialization_format="cloudpickle"
    )

print(f"Registered '{best_name}' pipeline as 'walmart-sales-xgboost-pipeline' in the MLflow Model Registry.")


2026/07/12 17:25:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Registered 'run11_train_stores_only' pipeline as 'walmart-sales-xgboost-pipeline' in the MLflow Model Registry.


Successfully registered model 'walmart-sales-xgboost-pipeline'.
Created version '1' of model 'walmart-sales-xgboost-pipeline'.
